In [2]:
# 12_cross_vendor_robustness.ipynb -- Cross-vendor robustness of the reconstruction and memorization findings.
# Reviewer concern (Round-1 Major 6): the three adversary tiers are one vendor family (OpenAI gpt-4o-mini /
# gpt-5.4-mini / gpt-5.4). Because the paper's headline is ABOUT model memorization, adversary identity is not
# incidental. This notebook re-runs, for one or more ADDITIONAL vendors, the two measurements that matter most:
#   (1) strong-tier identifier reconstruction at C2/C3/C4  (the P2/P3 trade-off), and
#   (2) the verbatim-overlap memorization probe            (the decontamination headline).
# It REUSES data/corpus.json, data/masked_corpus.json, data/id_maps.json from notebooks 01-02 (run those once
# first, no key needed). Everything is cached on disk keyed by (provider,model,...), so re-runs are free and
# adding a vendor never invalidates existing OpenAI results.
#
# HOW TO RUN
#   1) pip install -r requirements.txt   (only stdlib networking is used; numpy/pandas for stats)
#   2) Run notebooks 01 and 02 once (no key) so data/*.json exist.
#   3) Copy .env.example -> .env and fill in the vendor key(s) you have. You do NOT need all three vendors;
#      set the ones you have and list them in XVENDOR_PROVIDERS below.
#   4) Run this notebook. Outputs go to results/tables/xvendor_*.csv. Paths are not printed (anonymity).
#
# No proprietary data is used; the carrier is public Fineract + synthetic injected tokens.

import os, re, json, time, hashlib, difflib, urllib.request, urllib.error
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA  = ROOT / "data"
TAB   = ROOT / "results" / "tables"
CACHE = DATA / "_adv_cache"
for d in (TAB, CACHE):
    d.mkdir(parents=True, exist_ok=True)

# =========================================================================
# .env loader (repo root). Keys are never printed.
# =========================================================================
def load_env(p):
    cfg = {}
    if p.exists():
        for ln in p.read_text(encoding="utf-8").splitlines():
            ln = ln.strip()
            if ln and not ln.startswith("#") and "=" in ln:
                k, v = ln.split("=", 1)
                cfg[k.strip()] = v.strip().strip('"').strip("'")
    return cfg
ENV = load_env(ROOT / ".env")
def env(k, default=""):
    return ENV.get(k, os.environ.get(k, default))

# =========================================================================
# Which vendors/models to test. Add a provider here ONLY if you set its key in .env.
# Each entry: logical name -> (provider, model_id). The paper's own OpenAI tier is included
# for a within-run baseline; drop it if you only want the NEW vendors.
# =========================================================================
XVENDOR_MODELS = {
    # name           provider      model id (edit to the exact versions you have access to)
    "openai_strong":  ("openai",    env("LLM_MODEL_STRONG", "gpt-5.4")),
    "anthropic_strong": ("anthropic", env("ANTHROPIC_MODEL_STRONG", "claude-sonnet-4-5")),
    "google_strong":  ("google",    env("GOOGLE_MODEL_STRONG", "gemini-2.5-pro")),
}
# Only keep the providers whose key is present.
PROVIDER_KEYS = {
    "openai":    env("OPENAI_API_KEY"),
    "anthropic": env("ANTHROPIC_API_KEY"),
    "google":    env("GOOGLE_API_KEY") or env("GEMINI_API_KEY"),
}
ACTIVE = {name: (prov, mid) for name, (prov, mid) in XVENDOR_MODELS.items()
          if PROVIDER_KEYS.get(prov) and PROVIDER_KEYS[prov].lower() != "none"}

N_ADV_FILES = int(env("XVENDOR_N", "50"))   # files for the adversary (50 matches the paper)
K_IDENTS    = 12
BOOT        = 10000
NEAR        = 0.8

# =========================================================================
# Minimal multi-provider chat with on-disk caching. One deterministic call per prompt.
# Cache key includes provider+model so vendors never collide and OpenAI cache is reused as-is.
# =========================================================================
def _http_json(url, headers, body, timeout=120):
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

def _openai(model, prompt, temperature):
    body = {"model": model, "messages": [{"role": "user", "content": prompt}], "temperature": temperature}
    hdr = {"Authorization": f"Bearer {PROVIDER_KEYS['openai']}", "Content-Type": "application/json"}
    try:
        d = _http_json("https://api.openai.com/v1/chat/completions", hdr, body)
    except urllib.error.HTTPError as e:
        msg = e.read().decode(errors="ignore")
        if e.code == 400 and "temperature" in msg:       # some models reject non-default temperature
            body.pop("temperature"); d = _http_json("https://api.openai.com/v1/chat/completions", hdr, body)
        else:
            raise
    return d["choices"][0]["message"]["content"]

def _anthropic(model, prompt, temperature):
    body = {"model": model, "max_tokens": 1024, "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]}
    hdr = {"x-api-key": PROVIDER_KEYS["anthropic"], "anthropic-version": "2023-06-01",
           "Content-Type": "application/json"}
    d = _http_json("https://api.anthropic.com/v1/messages", hdr, body)
    # content is a list of blocks; take text blocks
    return "".join(b.get("text", "") for b in d.get("content", []) if b.get("type") == "text")

def _google(model, prompt, temperature):
    key = PROVIDER_KEYS["google"]
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={key}"
    body = {"contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": temperature}}
    d = _http_json(url, {"Content-Type": "application/json"}, body)
    cands = d.get("candidates", [])
    if not cands:
        return ""
    return "".join(p.get("text", "") for p in cands[0].get("content", {}).get("parts", []))

_DISPATCH = {"openai": _openai, "anthropic": _anthropic, "google": _google}

def chat(provider, model, prompt, temperature=0.0, tag=""):
    key = hashlib.sha256(f"{provider}|{model}|{temperature}|{tag}|{prompt}".encode()).hexdigest()
    cf = CACHE / f"xv_{key}.json"
    if cf.exists():
        return json.loads(cf.read_text(encoding="utf-8"))["content"]
    for attempt in range(4):                      # light retry for transient 429/5xx
        try:
            content = _DISPATCH[provider](model, prompt, temperature)
            cf.write_text(json.dumps({"content": content}), encoding="utf-8")
            return content
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503, 529) and attempt < 3:
                time.sleep(2 ** attempt); continue
            raise
    return ""

def parse_map(t):
    i, j = t.find("{"), t.rfind("}")
    if i == -1 or j == -1:
        return {}
    try:
        return json.loads(t[i:j + 1])
    except Exception:
        return {}
def norm(s):  return str(s).strip().strip("<>").lower()
def near(a, b): return difflib.SequenceMatcher(None, norm(a), norm(b)).ratio() >= NEAR
def boot_ci(v):
    a = np.asarray(v, float); a = a[~np.isnan(a)]
    if len(a) == 0: return (float("nan"), float("nan"))
    m = a[rng.integers(0, len(a), size=(BOOT, len(a)))].mean(axis=1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

# =========================================================================
# Load artifacts from notebooks 01-02
# =========================================================================
masked  = json.load(open(DATA / "masked_corpus.json", encoding="utf-8"))
id_maps = json.load(open(DATA / "id_maps.json", encoding="utf-8"))
corpus  = {c["file_id"]: c["code"] for c in json.load(open(DATA / "corpus.json", encoding="utf-8"))}
TIERS_ADV = ["C2", "C3", "C4"]            # C0/C1 are 1.0 by construction
masked_by_file = {t: {r["file_id"]: r["code"] for r in masked[t]} for t in masked}
sample_files = sorted(int(f) for f in id_maps.keys())[:N_ADV_FILES]

# =========================================================================
# (1) Identifier reconstruction at C2/C3/C4 (exact + near-match), per vendor
# =========================================================================
ADV_PROMPT = (
    "You are a security analyst testing whether masked source code leaks its original identifiers. "
    "Some identifiers were replaced by placeholder tokens of the form <ID_xxxxxx>. For each placeholder "
    "listed, infer the MOST LIKELY original identifier name from context. Return ONLY a JSON object "
    "mapping each placeholder to your single best guess, no explanation.\nPlaceholders: {ph}\n\nCode:\n{code}"
)
def reconstruct(provider, model, fid, tier, scoring):
    code = masked_by_file[tier][fid]
    idmap = id_maps.get(str(fid), {})
    present = [p for p in idmap if p in code][:K_IDENTS]
    if not present:
        return None
    g = parse_map(chat(provider, model, ADV_PROMPT.format(ph=present, code=code), 0.0, tag=f"recon_{tier}"))
    if scoring == "exact":
        return sum(1 for p in present if norm(g.get(p, "")) == norm(idmap[p])) / len(present)
    return sum(1 for p in present if near(g.get(p, ""), idmap[p])) / len(present)

# =========================================================================
# (2) Verbatim-overlap memorization probe (public carrier), per vendor
#     Mirrors nb09: show the first `head` lines of the ORIGINAL (unmasked) file, ask the model to continue,
#     measure overlap with the held-out next `head` lines.
# =========================================================================
def verbatim_overlap(provider, model, nf=12, head=8):
    rs = []
    for fid in sample_files[:nf]:
        lines = corpus[fid].splitlines()
        if len(lines) < head * 2:
            continue
        out = chat(provider, model,
                   "Continue this source file exactly as written. Output only the next lines of code.\n\n"
                   + "\n".join(lines[:head]), 0.0, tag="verbatim").splitlines()
        rs.append(difflib.SequenceMatcher(None, "\n".join(lines[head:head*2]), "\n".join(out[:head])).ratio())
    return float(np.mean(rs)) if rs else float("nan")

# =========================================================================
# Run
# =========================================================================
if not ACTIVE:
    print("No vendor keys found in .env. Set at least one of OPENAI_API_KEY / ANTHROPIC_API_KEY / GOOGLE_API_KEY,")
    print("list the model(s) in XVENDOR_MODELS, and re-run. (Notebooks 01-02 must have produced data/*.json.)")
else:
    recon_rows, mem_rows = [], []
    for name, (prov, mid) in ACTIVE.items():
        # memorization probe
        vb = verbatim_overlap(prov, mid)
        mem_rows.append({"adversary": name, "provider": prov, "model": mid, "verbatim_overlap": vb})
        # reconstruction per tier
        per_tier_exact = {}
        for tier in TIERS_ADV:
            ex = [reconstruct(prov, mid, f, tier, "exact") for f in sample_files]
            ex = [x for x in ex if x is not None]
            nm = [reconstruct(prov, mid, f, tier, "near") for f in sample_files]
            nm = [x for x in nm if x is not None]
            elo, ehi = boot_ci(ex); nlo, nhi = boot_ci(nm)
            per_tier_exact[tier] = np.array(ex, float)
            recon_rows.append({"adversary": name, "provider": prov, "model": mid, "tier": tier,
                               "recovery_exact": float(np.mean(ex)) if ex else float("nan"),
                               "exact_lo": elo, "exact_hi": ehi,
                               "recovery_near": float(np.mean(nm)) if nm else float("nan"),
                               "near_lo": nlo, "near_hi": nhi, "n": len(ex)})
        # paired C2 vs C3 on exact (the P2/P3 trade-off), per vendor
        c2, c3 = per_tier_exact["C2"], per_tier_exact["C3"]
        n = min(len(c2), len(c3))
        if n > 0:
            diff = c2[:n] - c3[:n]
            dl, dh = boot_ci(diff)
            try:
                from scipy import stats as st
                p = st.wilcoxon(c2[:n], c3[:n]).pvalue if np.any(diff != 0) else 1.0
            except Exception:
                p = float("nan")
            recon_rows.append({"adversary": name, "provider": prov, "model": mid, "tier": "C2-C3_paired",
                               "recovery_exact": float(diff.mean()), "exact_lo": dl, "exact_hi": dh,
                               "recovery_near": float("nan"), "near_lo": float("nan"), "near_hi": float("nan"),
                               "n": n, "wilcoxon_p": p})

    recon_df = pd.DataFrame(recon_rows)
    mem_df   = pd.DataFrame(mem_rows)
    recon_df.to_csv(TAB / "xvendor_reconstruction.csv", index=False)
    mem_df.to_csv(TAB / "xvendor_memorization.csv", index=False)

    print("=== Cross-vendor verbatim-overlap memorization probe (public carrier) ===")
    print(mem_df.to_string(index=False))
    print("\n=== Cross-vendor identifier reconstruction (exact + near-match) ===")
    show = recon_df[recon_df["tier"].isin(TIERS_ADV)][
        ["adversary", "tier", "recovery_exact", "exact_lo", "exact_hi", "recovery_near", "n"]]
    print(show.to_string(index=False))
    print("\n=== Per-vendor C2 vs C3 paired difference (exact; tests the P2/P3 trade-off) ===")
    pr = recon_df[recon_df["tier"] == "C2-C3_paired"][["adversary", "recovery_exact", "exact_lo", "exact_hi", "n", "wilcoxon_p"]]
    print(pr.to_string(index=False))
    print("\nRead: if verbatim_overlap is high for every vendor, memorization (not reconstruction) is a")
    print("cross-vendor property -> strengthens the decontamination headline. If the C2>C3 strong-adversary")
    print("trade-off replicates across vendors, P3 is not an OpenAI artifact.")


=== Cross-vendor verbatim-overlap memorization probe (public carrier) ===
    adversary provider   model  verbatim_overlap
openai_strong   openai gpt-5.4          0.997175

=== Cross-vendor identifier reconstruction (exact + near-match) ===
    adversary tier  recovery_exact  exact_lo  exact_hi  recovery_near  n
openai_strong   C2        0.630156  0.559956  0.697156       0.700498 50
openai_strong   C3        0.556303  0.496125  0.616836       0.656835 50
openai_strong   C4        0.557667  0.494857  0.619545       0.653351 50

=== Per-vendor C2 vs C3 paired difference (exact; tests the P2/P3 trade-off) ===
    adversary  recovery_exact  exact_lo  exact_hi  n  wilcoxon_p
openai_strong        0.073853  0.028604  0.119806 50     0.00227

Read: if verbatim_overlap is high for every vendor, memorization (not reconstruction) is a
cross-vendor property -> strengthens the decontamination headline. If the C2>C3 strong-adversary
trade-off replicates across vendors, P3 is not an OpenAI artifact.